In [3]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage

In [4]:
API = os.getenv("OPENROUTER_API_KEY")
MODEL = "nvidia/nemotron-3-nano-30b-a3b:free"
URL = "https://openrouter.ai/api/v1/"

client = ChatOpenAI(model=MODEL, api_key=API, base_url=URL)

In [5]:
class State(TypedDict):
    message: str
    sentiment: str
    topic: str
    response : str


def greet(state:State):
    return {"message": "Hello "}


def user(state:State):
    return {"message": state["message"] + "Miku!"}


def analyze_sentiment(state:State):
    prompt = f"Analyze the sentiment (positive/negative/neutral) of this text in one word: {state['message']}"
    result = client.invoke([HumanMessage(content=prompt)])
    return {"sentiment": result.content.strip()}


def extract_topic(state: State):
    prompt = f"Extract the main topic from this text in 2-3 words: {state['message']}"
    result = client.invoke([HumanMessage(content=prompt)])
    return {"topic": result.content.strip()}


def chatbot_response(state: State):
    messages = [
        HumanMessage(
            content=f"User said: {state['message']}. The sentiment is {state['sentiment']} and topic is {state['topic']}. Please respond naturally."
        )
    ]

    result = client.invoke(messages)

    return {"response": result.content}

In [6]:
workflow = StateGraph(State)

workflow.add_node("greet_node", greet)
workflow.add_node("user_node", user)
workflow.add_node("sentiment_node", analyze_sentiment)
workflow.add_node("topic_node", extract_topic)
workflow.add_node("chatbot_node", chatbot_response)

workflow.add_edge(START, "greet_node")
workflow.add_edge("greet_node", "user_node")
workflow.add_edge("user_node", "sentiment_node")
workflow.add_edge("sentiment_node", "topic_node")
workflow.add_edge("topic_node", "chatbot_node")
workflow.add_edge("chatbot_node", END)

app = workflow.compile()

In [7]:

input_state = {"message": "Hi", "sentiment": "", "topic": "", "response": ""}

output = app.invoke(input_state)

print(output)

{'message': 'Hello Miku!', 'sentiment': 'positive', 'topic': 'Hello Miku', 'response': "Hello! ✨ So glad yousaid hello! Ready to sing, dance, or chat? What's on your mind? 😄"}
